# MIPDS Milestone 5 — Detection and Spatial Features
**Aaron Tobias · CSC 5352 Deep Learning**

**Status: prepared and unexecuted; this is not a completed submission.**

This notebook implements the milestone rubric and builds on the supplied Chapter 5 hands-on notebook for MIPDS's temporary public exercise-recognition experiment. It uses the **actual Milestone 4 fine-tuned Keras model** for spatial-map inspection and representation comparison, and a **separate COCO-pretrained YOLOv8-nano** for object detection. YOLO does not read the fine-tuned ResNet features in this implementation.

Run the companion **Milestone 4 Public Exercise Recognition** notebook first. It supplies the real checkpoint, preprocessing record, exercise-labeled frame manifest, and existing grouped split. No heatmap observations, detection statistics, validation scores, or completed journal answers are supplied in this prepared file. A missing checkpoint or inconsistent run record stops execution rather than substituting weights from a different experiment.

The current experiment recognizes seven exercise types from side-view public recordings, using the publisher's clip labels. Participants remain disjoint across train/validation/test. This is a temporary recognition module while personal squat data are prepared; it does not establish squat phase, movement quality, or repetition counting.


## Inputs and run order

1. Finish both training stages, the prediction audit, and the journal in the companion Milestone 4 notebook.
2. Keep the same project folder (`/content/mipds_public_exercises` by default) with its frames, `classification_manifest.csv`, and `milestone4_outputs/m4_finetuned.keras` plus `m4_run_record.json`. If you mount Drive, set the same `ROOT` in both notebooks.
3. Milestone 4 prepares the three manifests below. Review their content and complete the two empty scene rows in `heatmap_manifest.csv` with real images: multiple people, and a person with another recognizable object. Manually verify the single-person row too.
4. In Colab, select a GPU runtime, run cells in order, and fill every response and journal cell from the rendered outputs. Use the supplied course notebook and rubric as the reference; another starter is not required to use this prepared workflow.

| Manifest | Required columns | Meaning |
|---|---|---|
| `classification_manifest.csv` | `path,label,split,client_id,session_id` | Reuse real Milestone 4 images, inherited exercise labels, and group assignments. |
| `heatmap_manifest.csv` | `image_id,path,scene_type,scene_note` | Exactly three distinct images: `single`, `same_class_multiple`, and `mixed_classes`. |
| `detection_manifest.csv` | `image_id,path,scene_note` | Exactly 20 distinct domain images with notes describing visible content. |

For the detection domain, use the rubric's **indoor scenes** option: home/gym exercise environments. Include challenging views, clutter, lighting differences, or occlusion when actually available; selecting only successful detections weakens the comparison. Milestone 4 defaults to its 20 audit frames, which may have limited scene diversity; you can deliberately replace this detector sample with 20 varied domain images. The classification split must remain unchanged. Specialized gym equipment may not have a COCO label.

Use recordings and other people's images with permission for course submission, and avoid identifiable bystanders. The detector's COCO labels form a separate label space from the squat-position classifier. The current classifier uses held-out public participants, with scope limited to this dataset and side-view setup; performance on personal/client recordings remains unmeasured.


In [ ]:
# Dependency setup. Colab normally includes TensorFlow; use the Milestone 4
# package versions when needed for checkpoint compatibility.
import importlib.util
import subprocess
import sys

packages = ["ultralytics>=8.3,<9", "scikit-learn>=1.4,<2", "pandas", "pillow", "matplotlib"]
if importlib.util.find_spec("tensorflow") is None:
    packages.append("tensorflow>=2.16,<3")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from IPython.display import display, Markdown
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ultralytics import YOLO

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

versions = {name: importlib.metadata.version(name) for name in
            ["tensorflow", "ultralytics", "scikit-learn", "numpy", "pandas", "pillow"]}
display(pd.Series(versions, name="installed_version"))


In [ ]:
# Same project folder as the companion Milestone 4 notebook.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path("/content/mipds_public_exercises")  # Or the same mounted Drive project folder used for M4.
M4_RECORD_PATH = ROOT / "milestone4_outputs" / "m4_run_record.json"
if not M4_RECORD_PATH.is_file():
    raise FileNotFoundError("Run the companion Milestone 4 notebook on the public exercise dataset first; keep its run record with the checkpoint.")
m4_record = json.loads(M4_RECORD_PATH.read_text(encoding="utf-8"))
required_record = {"milestone", "backbone", "nested_backbone", "feature_layer", "input_size",
                   "preprocessing", "preprocessing_location", "resize_method", "classes", "evaluation_scope",
                   "checkpoint_path", "checkpoint_sha256", "classification_manifest_sha256",
                   "frozen_weight_checks", "stages", "task", "dataset"}
if not required_record.issubset(m4_record):
    raise ValueError("Milestone 4 run record is incomplete; export it from the companion training notebook.")
if m4_record["milestone"] != 4 or len(m4_record["stages"]) != 2:
    raise ValueError("Require the completed two-stage Milestone 4 training record.")
if not all(m4_record["frozen_weight_checks"].get(key, False)
           for key in ["stage1", "stage2", "stage2_unfrozen_changed"]):
    raise ValueError("Milestone 4 weight-freezing checks did not pass.")
if m4_record["preprocessing_location"] != "outside_saved_model":
    raise ValueError("This companion adapter expects external preprocessing, as exported by Milestone 4.")

CHECKPOINT = ROOT / m4_record["checkpoint_path"]
CLASSIFICATION_CSV = ROOT / "classification_manifest.csv"
HEATMAP_CSV = ROOT / "heatmap_manifest.csv"
DETECTION_CSV = ROOT / "detection_manifest.csv"
OUTPUT = ROOT / "milestone5_outputs"
BACKBONE_FAMILY = m4_record["backbone"]
NESTED_BACKBONE_NAME = m4_record["nested_backbone"]
FEATURE_LAYER = m4_record["feature_layer"]
PREPROCESSING = m4_record["preprocessing"]
INPUT_SIZE = tuple(m4_record["input_size"])
RESIZE_METHOD = m4_record["resize_method"]
EVALUATION_SCOPE = m4_record["evaluation_scope"]
M4_CLASSES = m4_record["classes"]
M4_TASK = m4_record["task"]
M4_DATASET = m4_record["dataset"]
M4_SETTINGS_VERIFIED = False  # Becomes True after checkpoint/data hashes are verified below.
CUSTOM_OBJECTS = {}

BATCH_SIZE = 8
THRESHOLDS = (0.25, 0.50, 0.75)
YOLO_WEIGHTS = "yolov8n.pt"
YOLO_IMGSZ = 640
YOLO_IOU = 0.70
YOLO_DEVICE = "cpu"  # Set 0 to use the GPU; CPU reduces GPU memory competition.
PROBE_C = 1.0
PROBE_CLASS_WEIGHT = "balanced"
PROBE_MAX_ITER = 2000
MAX_FEATURE_MEMORY_MIB = 2048
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Using recorded M4 settings:", BACKBONE_FAMILY, PREPROCESSING, INPUT_SIZE, EVALUATION_SCOPE)


## Check the data and preserve the evaluation split

The classifier manifest must reuse Milestone 4's exact saved file, verified by its SHA-256 hash. Recording sessions, source videos, and identical image bytes cannot cross splits. For `unseen_people`, clients must also be disjoint. For `within_person_sessions`, the same person can appear in different sessions and splits, with conclusions limited to that person; pretending their sessions are different clients would give misleading evidence.

These checks cannot detect all near duplicates or incorrectly entered session IDs. Verify the actual recording boundaries. The approximate 70/15/15 design is not a reason to re-split the experiment here. Any real classification-data correction requires a new Milestone 4 run and matching record.

Missing detector/scene manifests get empty headers only. Complete them with real content; no replacement images or labels are generated. Milestone 4's defaults preserve existing detector and scene manifests on subsequent runs.


In [ ]:
schemas = {
    CLASSIFICATION_CSV: ["path", "label", "split", "client_id", "session_id"],
    HEATMAP_CSV: ["image_id", "path", "scene_type", "scene_note"],
    DETECTION_CSV: ["image_id", "path", "scene_note"],
}
for path, columns in schemas.items():
    if not path.exists():
        pd.DataFrame(columns=columns).to_csv(path, index=False)
        print(f"Created empty manifest: {path.name}; fill it with real data.")

def resolve_image_path(value):
    path = Path(value)
    return (ROOT / path).resolve() if not path.is_absolute() else path.resolve()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_manifest(path, required):
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(required) - set(table.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    if table.empty:
        raise ValueError(f"{path.name} is empty. Add your real images and metadata.")
    if table[required].apply(lambda col: col.str.strip().eq("")).any().any():
        raise ValueError(f"{path.name}: required fields cannot be blank.")
    table["resolved_path"] = table["path"].map(resolve_image_path)
    absent = [str(p) for p in table.resolved_path if not p.is_file()]
    if absent:
        raise FileNotFoundError(f"Missing image files (first three): {absent[:3]}")
    if table.resolved_path.duplicated().any():
        raise ValueError(f"{path.name}: image paths must be distinct.")
    table["sha256"] = table.resolved_path.map(sha256_file)
    return table

def validate_classification(table):
    splits = set(table.split)
    if not {"train", "validation"}.issubset(splits) or not splits.issubset({"train", "validation", "test"}):
        raise ValueError("Use train, validation, and optional test; keep the actual Milestone 4 assignments.")
    keys = ["session_id", "sha256"]
    if "video_path" in table.columns: keys.append("video_path")
    if EVALUATION_SCOPE == "unseen_people": keys.append("client_id")
    elif EVALUATION_SCOPE != "within_person_sessions": raise ValueError("Unknown M4 evaluation scope.")
    for key in keys:
        if (table.groupby(key)["split"].nunique() > 1).any():
            raise ValueError(f"Leakage: {key} crosses splits. Repair and rerun Milestone 4.")
    if "frame_id" in table.columns and table.frame_id.duplicated().any():
        raise ValueError("Classification frame IDs must be distinct.")
    synthetic_labels = {"retriever", "cat", "costume_dog", "scan_style"}
    if set(table.label).issubset(synthetic_labels):
        raise ValueError("The separate synthetic Chapter 4 exercise is not MIPDS milestone evidence.")
    train_labels = set(table.loc[table.split.eq("train"), "label"])
    val_labels = set(table.loc[table.split.eq("validation"), "label"])
    if len(train_labels) < 2 or not val_labels.issubset(train_labels):
        raise ValueError("Need at least two training classes and training coverage of every validation label.")

if sha256_file(CLASSIFICATION_CSV) != m4_record["classification_manifest_sha256"]:
    raise ValueError("Classification manifest differs from the one used to train Milestone 4.")
classification = read_manifest(CLASSIFICATION_CSV, schemas[CLASSIFICATION_CSV])
validate_classification(classification)
if set(classification.label) != set(M4_CLASSES):
    raise ValueError("Classification labels differ from Milestone 4's recorded label order.")
heatmap_images = read_manifest(HEATMAP_CSV, schemas[HEATMAP_CSV])
detector_images = read_manifest(DETECTION_CSV, schemas[DETECTION_CSV])
if len(heatmap_images) != 3 or set(heatmap_images.scene_type) != {"single", "same_class_multiple", "mixed_classes"}:
    raise ValueError("Stage 1 needs exactly one image for each of the three scene types.")
for name, table in [("Stage 1", heatmap_images), ("Stage 2", detector_images)]:
    if table.image_id.duplicated().any() or table.sha256.duplicated().any():
        raise ValueError(f"{name}: use distinct IDs and distinct image content.")
if len(detector_images) != 20:
    raise ValueError(f"Stage 2 requires exactly 20 images; found {len(detector_images)}.")

display(pd.crosstab(classification.label, classification.split))
display(detector_images[["image_id", "path", "scene_note"]])
print("No test images will be used to fit or evaluate the Stage 3 probes.")


## Load the real fine-tuned spatial extractor

The companion Milestone 4 model contains a nested ResNet-50. Its recorded `conv5_block3_out` layer supplies spatial maps before global averaging; with 224×224 input they are normally 7×7×2048. The code verifies the actual tensor. Keras ResNet preprocessing remains outside the model, exactly as in Milestone 4's validation pipeline.

The checkpoint hash and classification-manifest hash must match the completed training record. Load without compiling because this notebook uses a frozen extractor and new linear probes rather than resuming the optimizer. The model's BatchNormalization is kept in inference mode during extraction.


In [ ]:
if not CHECKPOINT.is_file():
    raise FileNotFoundError("Keep the actual fine-tuned Milestone 4 checkpoint in the shared project folder.")
if sha256_file(CHECKPOINT) != m4_record["checkpoint_sha256"]:
    raise ValueError("Checkpoint differs from the model exported in the Milestone 4 run record.")
saved_model = tf.keras.models.load_model(CHECKPOINT, compile=False, custom_objects=CUSTOM_OBJECTS)
M4_SETTINGS_VERIFIED = True
saved_model.summary()
print("Top-level layer names:", [layer.name for layer in saved_model.layers])


In [ ]:
if not M4_SETTINGS_VERIFIED or PREPROCESSING not in {"resnet50", "raw_0_255", "raw_0_1", "minus1_1"}:
    raise ValueError("Run the checkpoint and manifest verification cells; use the recorded Milestone 4 preprocessing.")
if BACKBONE_FAMILY not in {"resnet50", "efficientnetb0"}:
    raise ValueError("Use your actual ResNet-50 or EfficientNet-B0 backbone.")
feature_source = saved_model.get_layer(NESTED_BACKBONE_NAME) if NESTED_BACKBONE_NAME else saved_model
if not isinstance(feature_source, tf.keras.Model):
    raise TypeError("NESTED_BACKBONE_NAME must identify the trained backbone model.")
spatial_layer = feature_source.get_layer(FEATURE_LAYER)
spatial_model = tf.keras.Model(feature_source.inputs, spatial_layer.output)
spatial_model.trainable = False
if len(spatial_model.inputs) != 1 or len(spatial_model.output_shape) != 4:
    raise ValueError("Expected one input and a channel-last spatial feature output, before pooling.")
expected_channels = {"resnet50": 2048, "efficientnetb0": 1280}[BACKBONE_FAMILY]
if spatial_model.output_shape[-1] != expected_channels:
    raise ValueError("Wrong feature layer or tensor layout: check the last convolutional block.")
shape = spatial_model.input_shape
if len(shape) != 4 or shape[-1] != 3:
    raise ValueError("This adapter requires channel-last RGB input.")
for actual, configured in zip(shape[1:3], INPUT_SIZE):
    if actual is not None and actual != configured:
        raise ValueError("INPUT_SIZE does not match the saved extractor.")

def load_rgb(path):
    with Image.open(path) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))

def read_model_input(path):
    # Match the original validation resize, color handling, and normalization.
    # This implementation stretches to INPUT_SIZE without crop or letterbox.
    # Adapt this AND the heatmap alignment if Milestone 4 used different geometry.
    return tf.image.resize(load_rgb(path), INPUT_SIZE, method=RESIZE_METHOD).numpy().astype(np.float32)

def preprocess_batch(batch):
    batch = np.array(batch, dtype=np.float32, copy=True)
    if PREPROCESSING == "resnet50":
        return tf.keras.applications.resnet50.preprocess_input(batch)
    if PREPROCESSING == "raw_0_1":
        return batch / 255.0
    if PREPROCESSING == "minus1_1":
        return batch / 127.5 - 1.0
    return batch

def extract_maps(paths):
    batch = preprocess_batch(np.stack([read_model_input(p) for p in paths]))
    maps = np.asarray(spatial_model(batch, training=False), dtype=np.float32)
    if maps.ndim != 4 or not np.isfinite(maps).all():
        raise ValueError("Expected finite (batch, height, width, channels) maps.")
    if min(maps.shape[1:3]) < 4:
        raise ValueError("Spatial map must support the requested 4×4 pooling grid.")
    return maps

probe_map = extract_maps([heatmap_images.resolved_path.iloc[0]])
print("Verified feature tensor:", probe_map.shape)
print("Checkpoint SHA-256:", sha256_file(CHECKPOINT))


## Stage 1 — Spatial heatmaps for three scenes

Average each map over its channels and overlay it on the original image. These are **channel-mean activation maps**, not class-specific Grad-CAM explanations, causal evidence, or a measurement of an attention mechanism. Per-image min/max normalization makes spatial patterns visible, so color intensity should not be compared as an absolute activation score across different images. A flat activation map is displayed as flat, not as invented focus.


In [ ]:
def channel_mean_heatmap(feature_map):
    values = np.asarray(feature_map, dtype=np.float32).mean(axis=-1)
    low, high = float(values.min()), float(values.max())
    return np.zeros_like(values) if high - low <= 1e-8 else (values - low) / (high - low)

heatmap_files = []
scene_order = ["single", "same_class_multiple", "mixed_classes"]
for index, scene_type in enumerate(scene_order, start=1):
    row = heatmap_images.loc[heatmap_images.scene_type.eq(scene_type)].iloc[0]
    rgb = load_rgb(row.resolved_path)
    feature_map = extract_maps([row.resolved_path])[0]
    heat = channel_mean_heatmap(feature_map)
    aligned = np.asarray(Image.fromarray(heat, mode="F").resize(
        (rgb.shape[1], rgb.shape[0]), resample=Image.Resampling.BILINEAR))
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(rgb)
    axes[0].set_title("Original")
    axes[1].imshow(heat, cmap="inferno", vmin=0, vmax=1)
    axes[1].set_title(f"Mean activation · {feature_map.shape}")
    axes[2].imshow(rgb)
    axes[2].imshow(aligned, cmap="inferno", alpha=0.45, vmin=0, vmax=1)
    axes[2].set_title("Activation overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"Image {index} · {row.image_id} · {row.scene_note}")
    fig.tight_layout()
    figure_path = OUTPUT / f"stage1_image_{index}.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    heatmap_files.append(figure_path)
    plt.show()
    plt.close(fig)


### Stage 1 response — Image 1: single centered object
**[Write your observation after running the heatmap.]** Identify the exact person/body region or contextual object emphasized by the map. Does it align with the semantically meaningful region, or with clothing/background/setup cues? Describe the pattern as activation evidence, not proof of the model's reasoning.


### Stage 1 response — Image 2: multiple instances of one class
**[Write your observation after running the heatmap.]** Compare activation near every visible person or same-class instance. State whether one dominates, and explain how global averaging combines spatial positions without preserving which instance produced each response. Do not infer that all instances were detected from the heatmap alone.


### Stage 1 response — Image 3: complex scene with different classes
**[Write your observation after running the heatmap.]** Name the dominant regions and compare them with both visual salience and the real fine-tuning domain. Explain what the pattern suggests about domain adaptation, with uncertainty: one visualization cannot establish why a region is active.


## Stage 2 — YOLOv8-nano threshold experiment

Run all **20 images independently at each threshold**: 60 image/threshold records. Keep resolution, NMS IoU, and class handling fixed. Include zero-detection images in each average; their highest confidence is undefined, recorded as an empty value rather than zero. Record all detected COCO classes and also the number of `person` boxes because people are central to this application.

For qualitative assessment, inspect the rendered boxes for all 20 images at all three thresholds. Confidence alone does not establish correctness. No precision, recall, mAP, or probability calibration is claimed without appropriate ground-truth annotation/evaluation. Threshold choice needs the actual observed false boxes and missed people, especially occluded or distant clients.


In [ ]:
detector = YOLO(YOLO_WEIGHTS)
if detector.task != "detect" or len(detector.names) != 80 or "person" not in detector.names.values():
    raise ValueError("Use the required COCO-pretrained YOLOv8-nano detection checkpoint.")
print("YOLO classes:", detector.names)
if Path(YOLO_WEIGHTS).is_file():
    print("YOLO checkpoint SHA-256:", sha256_file(YOLO_WEIGHTS))

detector_records = []
boxes_payload = []
rendered_detections = {}
for threshold in THRESHOLDS:
    rendered_detections[threshold] = []
    for image_number, row in enumerate(detector_images.itertuples(index=False), start=1):
        result = detector.predict(
            source=load_rgb(row.resolved_path)[..., ::-1].copy(), conf=threshold, iou=YOLO_IOU,
            imgsz=YOLO_IMGSZ, device=YOLO_DEVICE, max_det=300,
            augment=False, agnostic_nms=False, verbose=False,
        )[0]
        boxes = result.boxes
        if boxes is None:
            confidences, class_ids, coordinates = np.array([]), np.array([], dtype=int), np.empty((0, 4))
        else:
            confidences = boxes.conf.cpu().numpy()
            class_ids = boxes.cls.cpu().numpy().astype(int)
            coordinates = boxes.xyxy.cpu().numpy()
        labels = [result.names[int(c)] for c in class_ids]
        detector_records.append({
            "image_id": row.image_id, "image_path": row.path,
            "scene_note": row.scene_note, "threshold": threshold,
            "detections": len(labels), "class_labels": ", ".join(labels) if labels else "none",
            "highest_confidence": float(confidences.max()) if len(confidences) else np.nan,
            "person_detections": labels.count("person"),
        })
        boxes_payload.append({
            "image_id": row.image_id, "threshold": threshold,
            "coordinate_system": "xyxy_pixels_in_exif_oriented_original_image",
            "image_height": int(result.orig_shape[0]), "image_width": int(result.orig_shape[1]),
            "source_model": "COCO-pretrained YOLOv8-nano; separate from M4 backbone",
            "boxes": [{"label": label, "confidence": float(conf), "xyxy": coord.tolist()}
                      for label, conf, coord in zip(labels, confidences, coordinates)],
        })
        # Ultralytics plotting returns BGR; convert to RGB for Matplotlib/Pillow.
        rendered = result.plot()[..., ::-1]
        rendered_path = OUTPUT / f"stage2_{threshold:.2f}_image_{image_number:02d}.jpg"
        Image.fromarray(rendered).save(rendered_path)
        rendered_detections[threshold].append((row.image_id, rendered_path))

detections = pd.DataFrame(detector_records)
assert len(detections) == 60
assert detections.groupby("threshold").size().eq(20).all()
detections.to_csv(OUTPUT / "stage2_per_image_detections.csv", index=False)
(OUTPUT / "stage2_boxes.json").write_text(json.dumps(boxes_payload, indent=2), encoding="utf-8")
for threshold in THRESHOLDS:
    display(Markdown(f"### Per-image results at confidence {threshold:.2f}"))
    display(detections.loc[detections.threshold.eq(threshold),
        ["image_id", "detections", "class_labels", "highest_confidence", "person_detections"]].reset_index(drop=True))


In [ ]:
# Inspect ALL 20 images for EACH threshold. Four-image pages remain readable.
for threshold in THRESHOLDS:
    images = rendered_detections[threshold]
    for start in range(0, len(images), 4):
        fig, axes = plt.subplots(2, 2, figsize=(12, 9))
        for ax, (image_id, path) in zip(axes.flat, images[start:start + 4]):
            ax.imshow(load_rgb(path))
            count = int(detections.loc[(detections.threshold.eq(threshold)) &
                        (detections.image_id.eq(image_id)), "detections"].iloc[0])
            ax.set_title(f"{image_id} · threshold {threshold:.2f} · {count} boxes")
            ax.axis("off")
        fig.tight_layout()
        plt.show()
        plt.close(fig)


In [ ]:
# Fill these SINGLE-SENTENCE assessments after reviewing the actual boxes.
QUALITATIVE_ASSESSMENT = {
    0.25: "",
    0.50: "",
    0.75: "",
}

def summarize_thresholds(records, qualitative):
    summary = records.groupby("threshold", sort=True).agg(
        images=("image_id", "nunique"),
        average_detections_per_image=("detections", "mean"),
        average_person_detections_per_image=("person_detections", "mean"),
        zero_detection_images=("detections", lambda values: int(values.eq(0).sum())),
    ).reset_index()
    summary["qualitative_assessment"] = summary.threshold.map(
        lambda threshold: qualitative.get(threshold, "").strip() or "[Write after visual review]"
    )
    return summary

threshold_summary = summarize_thresholds(detections, QUALITATIVE_ASSESSMENT)
display(threshold_summary.rename(columns={
    "threshold": "Threshold", "average_detections_per_image": "Avg. detections per image",
    "average_person_detections_per_image": "Avg. person boxes per image",
    "zero_detection_images": "Images with no detections", "qualitative_assessment": "Qualitative assessment",
}).round(3))
threshold_summary.to_csv(OUTPUT / "stage2_threshold_summary.csv", index=False)


### Stage 2 review notes
**[Write notes after inspecting the results.]** List specific image IDs where a low threshold added a false box or a high threshold removed a plausible person. Include content such as an occluded leg, a second person, a background reflection, or a cluttered chair **only if it is actually visible in your images/results**. State what you recommend and why. A smaller detection count by itself does not demonstrate improved accuracy.


## Stage 3 — Global versus multi-scale spatial pooling

Global average pooling produces `C` features. Pooling at 1×1, 2×2, and 4×4 grids produces `(1 + 4 + 16) × C = 21C` features: **43,008** for 2048-channel ResNet-50 or **26,880** for 1280-channel EfficientNet-B0. It summarizes coarse location rather than providing pixel-level segmentation, object identity, or precise box coordinates.

A 7×7 map does not divide evenly into a 4×4 grid. This scaffold uses adaptive average-pooling bin boundaries `floor(i×H/g)` to `ceil((i+1)×H/g)`, and the equivalent boundaries in width. Adjacent bins can overlap, following adaptive pooling semantics; no border cells are discarded. Flatten each grid in row/column/channel order and concatenate scales in the order 1, 2, 4. Replace this helper with the instructor-provided function if its bin convention differs and describe the convention actually used.


In [ ]:
def global_average_pool(maps):
    maps = np.asarray(maps, dtype=np.float32)
    if maps.ndim != 4 or not np.isfinite(maps).all():
        raise ValueError("Expected finite NHWC feature maps.")
    return maps.mean(axis=(1, 2), dtype=np.float32)

def adaptive_average_grid(maps, grid):
    maps = np.asarray(maps, dtype=np.float32)
    if maps.ndim != 4 or not np.isfinite(maps).all():
        raise ValueError("Expected finite NHWC feature maps.")
    n, height, width, channels = maps.shape
    if not isinstance(grid, int) or grid < 1 or min(height, width) < grid:
        raise ValueError("Grid must be a positive integer supported by the spatial map.")
    pooled = np.empty((n, grid, grid, channels), dtype=np.float32)
    for row in range(grid):
        top = math.floor(row * height / grid)
        bottom = math.ceil((row + 1) * height / grid)
        for col in range(grid):
            left = math.floor(col * width / grid)
            right = math.ceil((col + 1) * width / grid)
            pooled[:, row, col, :] = maps[:, top:bottom, left:right, :].mean(
                axis=(1, 2), dtype=np.float32)
    return pooled

def multi_scale_spatial_pool(maps, grids=(1, 2, 4)):
    maps = np.asarray(maps, dtype=np.float32)
    return np.concatenate([
        adaptive_average_grid(maps, grid).reshape(maps.shape[0], -1)
        for grid in grids
    ], axis=1)

gap_example = global_average_pool(probe_map)
spatial_example = multi_scale_spatial_pool(probe_map)
assert spatial_example.shape[1] == 21 * probe_map.shape[-1]
np.testing.assert_allclose(spatial_example[:, :probe_map.shape[-1]], gap_example)
print("Actual GAP dimension:", gap_example.shape[1])
print("Actual multi-scale dimension:", spatial_example.shape[1])
print("The first 1×1 block matches GAP.")


### Controlled linear-probe comparison

Freeze the **same trained backbone** and extract both representations from each identical input batch. Fit a separate standard scaler using **training features only**, followed by logistic regression with the same predeclared `C`, class weighting, iteration cap, and solver. Fit on the existing training split and evaluate the existing validation split once. The new probes are a representation comparison, not an exact reproduction of the Milestone 4 classifier's score or an additional fine-tuning stage.

The larger vector increases memory and classifier parameter count and may overfit a small dataset. Overall validation accuracy is required by the rubric; macro-F1 and balanced accuracy expose some effects of class imbalance. With grouped video data, image count is not the same as the number of independent clients/sessions. Neither this small comparison nor the 20-image detection exercise establishes deployment safety or guarantees 30 fps.


In [ ]:
train_rows = classification.loc[classification.split.eq("train")].reset_index(drop=True)
val_rows = classification.loc[classification.split.eq("validation")].reset_index(drop=True)
projected_bytes = (len(train_rows) + len(val_rows)) * expected_channels * 22 * 4
print(f"Pooled feature storage estimate: {projected_bytes / 1024**2:.1f} MiB (before scaler/solver copies).")
if projected_bytes / 1024**2 > MAX_FEATURE_MEMORY_MIB:
    raise MemoryError("Feature matrices exceed the configured budget; use a suitable runtime or memmap, preserving the split.")

def collect_pooled_features(rows):
    channels = expected_channels
    gap = np.empty((len(rows), channels), dtype=np.float32)
    multi = np.empty((len(rows), channels * 21), dtype=np.float32)
    for start in range(0, len(rows), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(rows))
        maps = extract_maps(rows.resolved_path.iloc[start:stop].tolist())
        gap[start:stop] = global_average_pool(maps)
        multi[start:stop] = multi_scale_spatial_pool(maps)
    return gap, multi

train_gap, train_spatial = collect_pooled_features(train_rows)
val_gap, val_spatial = collect_pooled_features(val_rows)
y_train = train_rows.label.to_numpy()
y_val = val_rows.label.to_numpy()
print("Train/validation GAP:", train_gap.shape, val_gap.shape)
print("Train/validation multi-scale:", train_spatial.shape, val_spatial.shape)


In [ ]:
def fit_linear_probe(train_features, validation_features, train_labels, validation_labels):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=PROBE_C, class_weight=PROBE_CLASS_WEIGHT,
                           max_iter=PROBE_MAX_ITER, solver="lbfgs", random_state=SEED),
    )
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(train_features, train_labels)
    if any(issubclass(item.category, ConvergenceWarning) for item in caught):
        raise RuntimeError("A probe did not converge. Adjust the common iteration cap, rerun both, and document the change.")
    predictions = model.predict(validation_features)
    classifier = model[-1]
    result = {
        "feature_dimensions": int(train_features.shape[1]),
        "classifier_parameters": int(classifier.coef_.size + classifier.intercept_.size),
        "train_accuracy": float(accuracy_score(train_labels, model.predict(train_features))),
        "validation_accuracy": float(accuracy_score(validation_labels, predictions)),
        "validation_macro_f1": float(f1_score(validation_labels, predictions, average="macro", zero_division=0)),
        "validation_balanced_accuracy": float(balanced_accuracy_score(validation_labels, predictions)),
        "fit_seconds": float(time.perf_counter() - started),
    }
    return model, predictions, result

probe_models, validation_predictions, comparison_records = {}, {}, []
for name, train_features, val_features in [
    ("Global average pooling", train_gap, val_gap),
    ("Multi-scale 1×1 + 2×2 + 4×4", train_spatial, val_spatial),
]:
    model, predictions, result = fit_linear_probe(train_features, val_features, y_train, y_val)
    probe_models[name], validation_predictions[name] = model, predictions
    comparison_records.append({"representation": name, **result})

comparison = pd.DataFrame(comparison_records)
display(comparison.round(4))
delta = comparison.validation_accuracy.iloc[1] - comparison.validation_accuracy.iloc[0]
print(f"Multi-scale minus GAP validation accuracy: {100 * delta:+.2f} percentage points.")
comparison.to_csv(OUTPUT / "stage3_pooling_comparison.csv", index=False)
label_order = sorted(set(y_train))
for name, predictions in validation_predictions.items():
    display(Markdown(f"**Validation confusion matrix — {name}** (rows = actual, columns = predicted)"))
    display(pd.DataFrame(confusion_matrix(y_val, predictions, labels=label_order),
                         index=label_order, columns=label_order))


### Stage 3 response — Representation tradeoff
**[Write after the comparison runs.]** Report both validation accuracies, their difference in percentage points, the actual channel count, and the actual feature dimensions. Explain what coarse spatial layout is retained and what remains lost. Discuss whether the observed change supports using the larger representation in this prototype, including train/validation behavior, model size, and the number of independent clients/sessions; an unchanged or worse score is a valid result.


## Stage 4 — Design journal

Replace the four response cells below with **3–5 sentences each**. Prompt 3 asks for two sentences about new scene expressiveness; use those as the first two sentences and add one or two sentences for the limitation. Ground the threshold answer in actual image IDs/results. Reference the actual Chapter 5 ethics section (Section 6); that reading is not attached here, and the references below are implementation documentation rather than a substitute for the assigned chapter.


### 1. Detection Architecture Justification
**[Write 3–5 sentences after reviewing Chapter 5.]** Recommend YOLO or Faster R-CNN for real-time assistive review in this domain and justify it using at least two chapter factors, such as inference latency/compute and detection quality for small or occluded people. State that 30+ fps is a deployment requirement to benchmark on the full pipeline rather than an observed result here. Describe one concrete condition that would change the recommendation, such as offline analysis where missed small objects matter more than latency.


### 2. Threshold Selection Rationale
**[Write 3–5 sentences after Stage 2.]** Select a threshold using the computed averages and visual review, identify actual images where 0.25 added distracting/spurious boxes or 0.75 missed a relevant person, and explain the effect on trainer review. If those failure types did not occur, describe what actually happened. Discuss the competing costs of missed clients and extra boxes; do not call a detector confidence score a calibrated probability of correctness.


### 3. What the System Can Now Express
**[Write 3–5 sentences.]** Use the first two sentences to explain the new COCO class/box outputs and coarse spatial feature layout compared with a single global vector. Then identify the most important remaining limitation for MIPDS: object boxes do not by themselves establish client identity, exercise phase, completed repetitions, joint geometry, or a validated coaching judgment. Explain the relevant integration boundary with MediaPipe and future temporal analysis without claiming those capabilities were implemented by this notebook.


### 4. Ethical Deployment Assessment
**[Write 3–5 sentences after reading Chapter 5, Section 6.]** Choose one domain-specific concern, such as identifying and locating a nonconsenting person in a client's home/gym recording. Explain the consequence and the relevant consent, cropping/redaction, access, or retention safeguard. Explicitly connect it to a concept actually discussed in Chapter 5's ethics section and provide the course's expected citation. Object detection does not identify named people, but visible faces and contextual details can still make the recording identifiable.


## Run record and downstream representation

Retain model provenance, input geometry, thresholds, the original split, and version information alongside results. Detector boxes are in oriented original-image pixels; the spatial summaries describe a separate resized classifier input. A downstream component should not silently equate their coordinates or treat either output as pose geometry. The example below exports the pooled features for one inspected image and matching detections only when that image was also in the 20-image detector sample.


In [ ]:
first = heatmap_images.loc[heatmap_images.scene_type.eq("single")].iloc[0]
first_map = extract_maps([first.resolved_path])
np.savez_compressed(OUTPUT / "example_scene_features.npz",
                    global_average=global_average_pool(first_map)[0],
                    multi_scale=multi_scale_spatial_pool(first_map)[0])
matching_detector_ids = detector_images.loc[detector_images.sha256.eq(first.sha256), "image_id"].tolist()
example_payload = {
    "scene_image_id": first.image_id,
    "image_sha256": first.sha256,
    "fine_tuned_backbone": BACKBONE_FAMILY,
    "spatial_map_shape_hwc": list(first_map.shape[1:]),
    "features_file": "example_scene_features.npz",
    "multi_scale_grids": [1, 2, 4],
    "feature_geometry": {"input_height_width": list(INPUT_SIZE), "resize": "stretch"},
    "detector_source": "independent COCO-pretrained YOLOv8-nano",
    "detection_results": [r for r in boxes_payload if r["image_id"] in matching_detector_ids],
    "detection_status": "included" if matching_detector_ids else "not run for this image",
    "limitations": ["no client identity", "no temporal repetition count", "no diagnostic or form judgment"],
}
(OUTPUT / "example_scene_payload.json").write_text(json.dumps(example_payload, indent=2), encoding="utf-8")

run_record = {
    "milestone": 5, "seed": SEED, "versions": versions,
    "milestone4_record_sha256": sha256_file(M4_RECORD_PATH),
    "evaluation_scope": EVALUATION_SCOPE, "classes": M4_CLASSES,
    "milestone4_task": M4_TASK, "milestone4_dataset": M4_DATASET,
    "checkpoint_sha256": sha256_file(CHECKPOINT),
    "backbone": BACKBONE_FAMILY, "nested_backbone": NESTED_BACKBONE_NAME,
    "feature_layer": FEATURE_LAYER, "input_size": list(INPUT_SIZE),
    "preprocessing": PREPROCESSING, "resize_method": RESIZE_METHOD,
    "pooling": "adaptive floor/ceil average bins; NHWC flatten; scales 1,2,4",
    "linear_probe": {"algorithm": "logistic regression", "C": PROBE_C,
                     "class_weight": PROBE_CLASS_WEIGHT, "max_iter": PROBE_MAX_ITER,
                     "standardizer_fit": "train only"},
    "split_sizes": classification.split.value_counts().to_dict(),
    "unique_clients_per_split": classification.groupby("split").client_id.nunique().to_dict(),
    "unique_sessions_per_split": classification.groupby("split").session_id.nunique().to_dict(),
    "manifest_sha256": {p.name: sha256_file(p) for p in schemas},
    "detector": {"weights": YOLO_WEIGHTS, "confidence_thresholds": list(THRESHOLDS),
                 "iou": YOLO_IOU, "imgsz": YOLO_IMGSZ, "device": str(YOLO_DEVICE)},
    "claim_boundary": "validation representation comparison and qualitative 20-image detection analysis",
}
(OUTPUT / "run_record.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print("Saved the real run's figures, tables, detector boxes, example features, and provenance.")


In [ ]:
checks = {
    "Three heatmap figure files": len(heatmap_files) == 3 and all(p.is_file() for p in heatmap_files),
    "20 images at each of 3 thresholds": len(detections) == 60 and detections.groupby("threshold").size().eq(20).all(),
    "Qualitative sentences entered": all(QUALITATIVE_ASSESSMENT[t].strip() for t in THRESHOLDS),
    "Both validation accuracies computed": len(comparison) == 2 and comparison.validation_accuracy.notna().all(),
    "Correct multi-scale dimension": spatial_example.shape[1] == expected_channels * 21,
    "Run provenance exported": (OUTPUT / "run_record.json").is_file(),
}
display(pd.DataFrame([{"check": key, "passed": bool(value)} for key, value in checks.items()]))
print("Manual review still required: replace every response/journal placeholder and cite Chapter 5 Section 6.")
print("After edits, restart the runtime and run all cells. Resolve every error before downloading.")
print("In Colab: File → Download → Download .ipynb; retain rendered outputs in the submitted notebook.")


## Submission review

- [ ] The supplied course notebook and milestone rubric have been checked against this implementation.
- [ ] The checkpoint and classification data are from the real Milestone 4 experiment.
- [ ] All three heatmaps and their response cells are complete.
- [ ] All 20 per-image records and the three-threshold summary are rendered; the qualitative sentences match visual review.
- [ ] The GAP/multi-scale validation accuracy comparison is rendered on the same original split.
- [ ] All four journal answers contain 3–5 sentences; the ethics answer cites the assigned Chapter 5 Section 6.
- [ ] Identifiable client/bystander information is handled consistently with permission for course submission.
- [ ] A clean run finishes without errors and the downloaded `.ipynb` retains outputs.
- [ ] Follow the course syllabus's AI-use and disclosure requirements.

**This scaffold does not satisfy these items until the real experiment has been run and the responses completed.**

## Implementation references

These sources support the code choices, not any unrun experiment or a substitute citation to your textbook:

- [Keras ResNet models: preprocessing and spatial versus pooled outputs](https://keras.io/api/applications/resnet/resnet_models/)
- [Keras EfficientNet: internal preprocessing and input range](https://keras.io/api/applications/efficientnet/efficientnet_models/)
- [Keras transfer learning and inference-mode feature extraction](https://keras.io/guides/transfer_learning/)
- [Ultralytics YOLOv8 model family](https://docs.ultralytics.com/models/yolov8/)
- [Ultralytics prediction settings and result boxes](https://docs.ultralytics.com/modes/predict/)
- [PyTorch adaptive average pooling convention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.adaptive_avg_pool2d.html)
- [scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Public exercise dataset: Mendeley Data V3, CC BY 4.0, DOI 10.17632/kgbb3yn47p.3](https://data.mendeley.com/datasets/kgbb3yn47p/3)
- Assigned course Chapter 5, Section 6: **add the actual chapter citation after reading it**.
